[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

# Lesson 10 — Exceptions

**Module 1 — Python Fundamentals** | ⏱ 20 min

Exceptions are Python's mechanism for handling errors and unexpected situations gracefully. Instead of crashing when something goes wrong, a well-written Python program catches exceptions and responds intelligently — retrying, logging the error, or returning a sensible default. Understanding the exception hierarchy and how to raise meaningful exceptions is a key professional programming skill.

## Learning Objectives
- Use the full `try/except/else/finally` structure
- Catch specific exception types rather than using bare `except`
- Understand Python's exception hierarchy
- Raise exceptions with descriptive messages using `raise`
- Chain exceptions with `raise ... from ...`
- Create custom exception classes with additional attributes

## try / except / else / finally

The complete exception handling structure has four optional clauses. `try` contains the code that might raise an exception. `except` catches and handles specific exceptions. `else` runs only if no exception was raised (the happy path). `finally` always runs, regardless of whether an exception occurred — it is the right place for cleanup code like closing files or releasing resources.

In [ ]:
# Basic try/except structure
def safe_divide(a, b):
    """Divide a by b, returning None if b is zero."""
    try:
        result = a / b           # This might raise ZeroDivisionError
    except ZeroDivisionError:
        print(f"Error: Cannot divide {a} by zero.")
        return None
    else:
        # Only runs if NO exception was raised in try
        print(f"Success: {a} / {b} = {result:.4f}")
        return result
    finally:
        # ALWAYS runs — for cleanup
        print(f"--- divide({a}, {b}) completed ---")

print(safe_divide(10, 3))
print()
print(safe_divide(10, 0))

In [ ]:
# The else clause vs putting code at the end of try — what's the difference?

# Problem with this approach: if process_data() raises, we catch it too
def process_file_naive(filepath):
    try:
        with open(filepath) as f:
            data = f.read()
        result = data.upper()  # Bug: if open() failed we'd never get here,
                               # but if THIS raises, our except catches it too
    except FileNotFoundError:
        return "File not found"
    return result

# Better: use else to run code only when try succeeded
def process_file_better(filepath):
    try:
        with open(filepath) as f:
            data = f.read()
    except FileNotFoundError:
        return "File not found"  # Only handles file-not-found
    else:
        # Only runs if open() succeeded — exceptions here are NOT caught above
        return data.upper()

print(process_file_better("sample_notes.txt")[:50])
print(process_file_better("nonexistent.txt"))

## Catching Specific Exceptions

Always catch the most specific exception you expect. Using a bare `except:` or `except Exception:` is a bad practice because it silences bugs you did not anticipate. Python has a rich hierarchy of built-in exceptions — knowing the common ones helps you write precise handlers. You can catch multiple exception types in one `except` clause using a tuple, and you can access the exception object with `as e` to get its message and attributes.

In [ ]:
# Common built-in exceptions
test_cases = [
    ("valid",     lambda: int("42")),
    ("not a num", lambda: int("hello")),         # ValueError
    ("type err",  lambda: "text" + 5),            # TypeError
    ("key err",   lambda: {"a": 1}["b"]),         # KeyError
    ("index err", lambda: [1, 2, 3][10]),          # IndexError
    ("attr err",  lambda: (42).upper()),           # AttributeError
    ("zero div",  lambda: 1 / 0),                 # ZeroDivisionError
    ("overflow",  lambda: float('inf') * 0),      # Not overflow but NaN
]

for label, operation in test_cases:
    try:
        result = operation()
        print(f"  {label:<15}: {result}")
    except ValueError as e:
        print(f"  {label:<15}: ValueError — {e}")
    except TypeError as e:
        print(f"  {label:<15}: TypeError — {e}")
    except KeyError as e:
        print(f"  {label:<15}: KeyError — key {e} not found")
    except IndexError as e:
        print(f"  {label:<15}: IndexError — {e}")
    except (AttributeError, ZeroDivisionError) as e:
        # Catch multiple types in one clause
        print(f"  {label:<15}: {type(e).__name__} — {e}")

In [ ]:
# Exception hierarchy — understanding parent/child relationships
# BaseException
#   SystemExit, KeyboardInterrupt, GeneratorExit  (don't catch these!)
#   Exception
#     ArithmeticError
#       ZeroDivisionError, OverflowError, FloatingPointError
#     LookupError
#       IndexError, KeyError
#     ValueError, TypeError, AttributeError, RuntimeError, OSError...
#       OSError: FileNotFoundError, PermissionError, IsADirectoryError...

# Catching a parent type catches all children
def lookup(container, key):
    try:
        return container[key]
    except LookupError as e:
        # Catches BOTH IndexError and KeyError (both are LookupError children)
        print(f"Lookup failed ({type(e).__name__}): {e}")
        return None

print(lookup([1, 2, 3], 10))         # IndexError caught
print(lookup({"a": 1}, "b"))        # KeyError caught
print(lookup({"a": 1}, "a"))        # Success — returns 1

# Inspect the exception object
try:
    result = 10 / 0
except ZeroDivisionError as e:
    print(f"\nException type: {type(e).__name__}")
    print(f"Exception args: {e.args}")
    print(f"Exception message: {str(e)}")

## Raising Exceptions

You can raise exceptions intentionally using the `raise` statement to signal that something has gone wrong in your code. Always provide a clear, descriptive message. A function should raise an exception when it receives invalid input rather than silently producing wrong results. The `raise` statement without arguments inside an `except` block **re-raises** the current exception, preserving the original traceback.

In [ ]:
# Raising exceptions with descriptive messages
def set_age(age):
    """Validate and return a user's age."""
    if not isinstance(age, int):
        raise TypeError(f"Age must be an integer, got {type(age).__name__}")
    if age < 0:
        raise ValueError(f"Age cannot be negative: {age}")
    if age > 150:
        raise ValueError(f"Age {age} is implausibly large (max: 150)")
    return age

# Test various inputs
test_values = [25, -5, 200, 3.14, "old"]
for value in test_values:
    try:
        result = set_age(value)
        print(f"  set_age({value!r}) = {result}")
    except (TypeError, ValueError) as e:
        print(f"  set_age({value!r}) FAILED: {type(e).__name__}: {e}")

In [ ]:
# Raising inside except — re-raise and raise from
import json

# Re-raise: preserve the original exception and traceback
def parse_user_json(json_str):
    try:
        data = json.loads(json_str)
        return data
    except json.JSONDecodeError:
        print("Logging: JSON parsing failed")  # Log it
        raise  # Re-raise the SAME exception — don't lose the traceback

try:
    parse_user_json('{"name": invalid}')  # Bad JSON
except json.JSONDecodeError as e:
    print(f"Caught after re-raise: {e}")

print()

# raise ... from ... : exception chaining (PEP 3134)
def fetch_user_from_db(user_id):
    fake_db = {1: "alice", 2: "bob"}
    try:
        return fake_db[user_id]
    except KeyError as original_error:
        # Wrap the low-level error in a more meaningful high-level error
        raise ValueError(f"User with ID {user_id} not found") from original_error

try:
    fetch_user_from_db(99)
except ValueError as e:
    print(f"High-level error: {e}")
    print(f"Caused by: {e.__cause__}")  # The original KeyError

## Custom Exception Classes

For larger applications, creating custom exception classes makes error handling much more expressive and precise. Custom exceptions are classes that inherit from `Exception` (or a more specific built-in exception). You can add custom attributes to carry additional context about the error — for example, an HTTP status code, an error code, or the specific data that caused the problem.

In [ ]:
# Custom exception hierarchy for a payment processing system

class PaymentError(Exception):
    """Base class for all payment-related errors."""
    pass

class InsufficientFundsError(PaymentError):
    """Raised when an account does not have enough balance."""
    def __init__(self, requested, available, account_id):
        self.requested = requested
        self.available = available
        self.account_id = account_id
        self.shortfall = requested - available
        super().__init__(
            f"Account {account_id}: requested ${requested:.2f}, "
            f"only ${available:.2f} available (shortfall: ${self.shortfall:.2f})"
        )

class InvalidCardError(PaymentError):
    """Raised when a card number is invalid."""
    def __init__(self, card_number, reason):
        self.card_number = f"****{card_number[-4:]}"  # Mask for security
        self.reason = reason
        super().__init__(f"Card {self.card_number} invalid: {reason}")

class PaymentLimitExceededError(PaymentError):
    """Raised when a single transaction exceeds the allowed limit."""
    DAILY_LIMIT = 5000.00

    def __init__(self, amount):
        self.amount = amount
        super().__init__(
            f"Transaction of ${amount:.2f} exceeds daily limit of ${self.DAILY_LIMIT:.2f}"
        )

In [ ]:
# Using custom exceptions
def process_payment(account_id, card_number, amount, balance):
    """Process a payment, raising descriptive exceptions on failure."""
    # Validate card (simplified)
    if len(card_number) != 16 or not card_number.isdigit():
        raise InvalidCardError(card_number, "must be 16 digits")

    # Check limit
    if amount > PaymentLimitExceededError.DAILY_LIMIT:
        raise PaymentLimitExceededError(amount)

    # Check balance
    if amount > balance:
        raise InsufficientFundsError(amount, balance, account_id)

    return {"status": "success", "amount": amount, "account": account_id}

# Test scenarios
test_payments = [
    ("ACC001", "4532015112830366", 150.00, 500.00),   # Success
    ("ACC001", "4532015112830366", 600.00, 200.00),   # Insufficient funds
    ("ACC002", "4532015112830366", 7500.00, 10000.00), # Exceeds limit
    ("ACC003", "invalid-card",     100.00, 500.00),   # Bad card
]

for acc, card, amount, balance in test_payments:
    try:
        result = process_payment(acc, card, amount, balance)
        print(f"Payment OK: {result}")
    except InsufficientFundsError as e:
        print(f"INSUFFICIENT FUNDS: {e}")
        print(f"  -> Shortfall: ${e.shortfall:.2f}")
    except PaymentLimitExceededError as e:
        print(f"LIMIT EXCEEDED: {e}")
    except InvalidCardError as e:
        print(f"INVALID CARD ({e.card_number}): {e.reason}")
    except PaymentError as e:
        # Catch-all for any other payment errors (shouldn't happen here)
        print(f"PAYMENT ERROR: {e}")

## Practice Exercises

1. Write a function `safe_json_parse(text)` that attempts to parse a JSON string and returns a tuple `(success, data)` — where `success` is `True` and `data` is the parsed object on success, or `False` and an error message string on failure. Handle `json.JSONDecodeError` and `TypeError`.
2. Create a custom exception `ConfigurationError` with attributes `key` (the missing/invalid config key) and `expected_type`. Write a function `load_settings(config_dict)` that validates required keys and raises `ConfigurationError` for any missing or wrong-typed keys.
3. Write a function `retry(func, max_attempts, exceptions)` that calls `func()` up to `max_attempts` times. If `func` raises any exception in `exceptions`, it waits (use `time.sleep(0)` for the test) and retries. If all attempts fail, it re-raises the last exception. Test it with a function that fails the first two times then succeeds.
4. Rewrite this code to use `try/except/else/finally` correctly: a function that opens a file, parses each line as a float, returns the average — logging a message whether it succeeded or failed, and always closing the file.